## Lib integration

In [1]:
import sys
# sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching")
sys.path.append(r"E:\Dai hoc\2526I\dacn\flow-matching\demo-code\2d")
import h5py
from collections import defaultdict, Counter
import numpy as np
from rich import print

In [2]:
file_path = r"E:\Dai hoc\2526I\dacn\flow-matching\data\traintest_hcd.hdf5"

## Open data

In [3]:
with h5py.File(file_path, "r") as f:
    print("Keys:", list(f.keys()))
    big_batch = 10000
    seqs = f["sequence_integer"][:big_batch]
    charges_oh = f["precursor_charge_onehot"][:big_batch]
    intensities = f["intensities_raw"][:big_batch]  

Keys:
[
    'collision_energy',
    'collision_energy_aligned',
    'collision_energy_aligned_normed',
    'intensities_raw',
    'masses_pred',
    'masses_raw',
    'method',
    'precursor_charge_onehot',
    'rawfile',
    'reverse',
    'scan_number',
    'score',
    'sequence_integer',
    'sequence_onehot'
]

In [4]:
charges = np.argmax(charges_oh, axis=1) + 1
del charges_oh

### Utils

### Peak data

In [5]:
# np.argmax(charges_oh, axis=1) + 1

In [6]:
len(intensities[0])

174

In [7]:
max_index = 0
for seq in seqs:
    for token in seq:
        if token > max_index:
            max_index = token
print("Max token index:", max_index)

Max token index: 21

## Explore data

## FLow matching training

In [8]:
import torch
torch.set_default_device("cuda")
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
import imageio
from sklearn.datasets import make_moons
import math

from coupling import mini_batch_coupling, greedy_coupling, sinkhorm_coupling
from draw import DrawFlow, DrawSample
from gen_path import get_xt

In [9]:
class CFGFlow(nn.Module):
    def __init__(self, noise_dim, *cond_dim):
        super().__init__()
        self.noise_dim = noise_dim
        # self.cond_dim = cond_dim

    def forward(self, x: torch.Tensor, t: torch.Tensor, **cond):
        raise NotImplementedError("CFG Model is not implemented")
    
    def step(self, x_t: torch.Tensor, cond:torch.Tensor, t_start: torch.Tensor, t_end: torch.Tensor):
        raise NotImplementedError("CFG Model is not implemented")
    
    def sample(self, cond: torch.Tensor, batch: int = 32, step:int = 10):
        x_t = torch.randn(batch, self.noise_dim, device=cond.device)
        t = torch.zeros(batch, device=cond.device)
        dt = 1.0 / step
        for _ in range(step):
            t_start = t
            t_end = t + dt
            x_t = self.step(x_t, cond, t_start, t_end)
            t = t_end
        return x_t

In [14]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden_dim=256, layers=4):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(input_dim, hidden_dim))
        for _ in range(layers - 1):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
            self.layers.append(nn.SiLU())
        self.output_layer = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.output_layer(x)

In [15]:
class HCDFlow(CFGFlow):
    pep_dim = 128
    time_dim = 32
    charge_dim = 32
    def __init__(self, noise_dim, pep_dim=128, time_dim=32, charge_dim=32):
        super().__init__(noise_dim)
        self.noise_dim = noise_dim
        self.pep_dim = pep_dim
        self.time_dim = time_dim
        self.charge_dim = charge_dim
        self.pep_embedding = nn.Embedding(22, self.pep_dim, padding_idx=0)
        self.pos_embedding = nn.Parameter(torch.randn(1, 30, pep_dim))
        self.charge_embedding = nn.Linear(1, self.charge_dim)
        self.time_embedding = nn.Linear(1, self.time_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=pep_dim, 
            nhead=8, 
            dim_feedforward=512,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        total_cond_dim = self.pep_dim + self.charge_dim + self.time_dim
        self.net = MLP(input_dim=total_cond_dim + noise_dim, output_dim=noise_dim, hidden_dim=1024, layers=4)
    
    def forward(self, x: torch.Tensor, t: torch.Tensor, pep_seq, charge):
        time_embs = self.time_embedding(t)
        charge_embs = self.charge_embedding(charge)
        
        pep_embs = self.pep_embedding(pep_seq) + self.pos_embedding
        
        p_out = self.transformer(pep_embs)
        p_cond = p_out.mean(dim=1)
        cond = torch.cat([time_embs, p_cond, charge_embs], dim=-1)
        
        return self.net(torch.cat([x, cond], dim=-1))
    
    def step(self, x_t: torch.Tensor, pep_seq, charge, t_start: torch.Tensor, t_end: torch.Tensor):
        # Use midpoint method for better accuracy
        t_mid = (t_start + t_end) / 2
        # x_next = x + f(x+ f(x, t_start) * (t_end - t_start) / 2, t_mid)
        v_x = self(x_t + self(x_t, t_start, pep_seq, charge) * (t_end - t_start) / 2, t_mid, pep_seq, charge)
        x_next = x_t + v_x * (t_end - t_start)
        return x_next

### Training configuration

In [16]:
epoch = 1e6
batch_size = 1024
model = HCDFlow(174)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-10)

In [18]:
loss_history = []
for ep in range(int(epoch)):
    model.train()
    batch_indices = np.random.choice(len(intensities), batch_size, replace=False)
    batch_intensities = torch.tensor(intensities[batch_indices], dtype=torch.float32)
    batch_pep_seq = torch.tensor(seqs[batch_indices], dtype=torch.int16)
    batch_charge = torch.tensor(charges[batch_indices], dtype=torch.float32).unsqueeze(1)
    
    noise = torch.randn_like(batch_intensities)
    t = torch.rand(batch_size, 1)
    x_t = get_xt(batch_intensities, noise, t)
    
    pred_noise = model(x_t, t, pep_seq=batch_pep_seq, charge=batch_charge)
    
    loss = nn.MSELoss()(pred_noise, noise)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if ep % 100 == 0:
        print(f"Epoch {ep}, Loss: {loss.item():.4f}")



Epoch 0, Loss: 1.0012

Epoch 100, Loss: 1.0008

Epoch 200, Loss: 0.9972

Epoch 300, Loss: 0.9965

Epoch 400, Loss: 0.9975

...

Epoch 600, Loss: 1.0007

Epoch 700, Loss: 0.9977

Epoch 800, Loss: 0.9981

Epoch 900, Loss: 0.9948

KeyboardInterrupt: 